In [100]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from utils.normal_features import *

PROJECT_ROOT = Path.cwd().resolve().parent.parent
DATA_EXPLODED_PATH = PROJECT_ROOT / 'data' / 'exploded_splits'

TRAIN_PATH = DATA_EXPLODED_PATH / "train_pairs.parquet"
VAL_PATH = DATA_EXPLODED_PATH / "validation_pairs.parquet"
TEST_PATH = DATA_EXPLODED_PATH / "test_pairs.parquet"

OUTPUT_DIR =  PROJECT_ROOT / 'data' / 'normal_features'
ID_REGISTRY_PATH = PROJECT_ROOT / 'data' / "ids_registry.parquet"

In [77]:
# load the file 
train = pd.read_parquet(TRAIN_PATH)
val = pd.read_parquet(VAL_PATH)
test = pd.read_parquet(TEST_PATH)

# Feature engineering: normal features

## 1. Exploration
Briefly exploration of the columns and the type.

In [78]:
train.head(2)

,article_id,ref_id,title_article,lang_article,doc_type_article,abstract_article,year_article,n_citation_article,keywords_article,authors_article,title_ref,lang_ref,doc_type_ref,abstract_ref,year_ref,keywords_ref,authors_ref,is_reference_valid,split
0,53e99f0ab7602d97027d6a89,53e99ff0b7602d97028d14d3,Comments on a Method of Karpovsky,en,journal,Karpovsky has presented methods of multiple-va...,1978.0,12.0,"[karpovsky, method]","[{'id': '53f44aa3dabfaee43ec8ec34', 'name': 'C...",Harmonic-Analysis over Finite Commutative Grou...,en,journal,In this paper we consider the linearization pr...,1977.0,None,"[{'id': '562c496445cedb3398bde732', 'name': 'M...",1,train
1,53e9bd81b7602d9704a24d06,557f4d4f6fee0fe990cb035f,The Effect of Visual Information on Word Initi...,en,conference,Disabled individuals can realize many benefits...,1996.0,5.0,"[audio-visual systems, handicapped aids, heari...","[{'id': '53f44a07dabfaeb22f4cee3a', 'name': 'R...",Improving Connected Letter Recognition by Lipr...,en,conference,In this paper we show how recognition performa...,1993.0,"[error rate, connected word recognition proble...","[{'id': '5484d790dabfae9b40133197', 'name': 'C...",1,train


## 2. Filter
First of all not all the features are usefull for our project, we should select the ones that can be used.


In [79]:
train.columns

Index(['article_id', 'ref_id', 'title_article', 'lang_article',
       'doc_type_article', 'abstract_article', 'year_article',
       'n_citation_article', 'keywords_article', 'authors_article',
       'title_ref', 'lang_ref', 'doc_type_ref', 'abstract_ref', 'year_ref',
       'keywords_ref', 'authors_ref', 'is_reference_valid', 'split'],
      dtype='str')

In [80]:
cols_to_exclude = ['article_id', 'ref_id', 'title_article', 'abstract_article', 'title_ref', 'abstract_ref', 'split']

for df in [train, val, test]:
    df.drop(cols_to_exclude, inplace=True, axis=1)

In [81]:
train.columns

Index(['lang_article', 'doc_type_article', 'year_article',
       'n_citation_article', 'keywords_article', 'authors_article', 'lang_ref',
       'doc_type_ref', 'year_ref', 'keywords_ref', 'authors_ref',
       'is_reference_valid'],
      dtype='str')

## 3. Feature Engineering
Since usually the models cannot use categorical or string columns, the column should be converted in a model readable format.

In [82]:
display(train.dtypes)

lang_article              str
doc_type_article          str
year_article          float64
n_citation_article    float64
keywords_article       object
authors_article        object
lang_ref                  str
doc_type_ref              str
year_ref              float64
keywords_ref           object
authors_ref            object
is_reference_valid      int64
dtype: object

### 3.1 Add some features
Before modify, add some useful columns regarding the columns that we are going to modify.

Since keywords and authors are structured data as `np.ndarrays`, before embed them we will create the columns that count how many of them are in the paper.

We include `n_keywords` and `n_authors` to provide the model with quantitative context regarding the metadata density of the papers.

In [83]:
for df in [train, val, test]:
    # add n_keywords_article and n_keywords_ref
    # add n_authors_article and n_authors_ref
    for col_to_sum in ['keywords_article', 'keywords_ref', 'authors_article', 'authors_ref']:
        counter_series = df[col_to_sum].apply(lambda obj: 0 if not isinstance(obj, np.ndarray) else len(obj))
        df['n_'+col_to_sum] = counter_series

In [84]:
train[['keywords_article', 'keywords_ref', 'n_keywords_article', 'n_keywords_ref', 'authors_article', 'authors_ref', 'n_authors_article', 'n_authors_ref']].head(2)

,keywords_article,keywords_ref,n_keywords_article,n_keywords_ref,authors_article,authors_ref,n_authors_article,n_authors_ref
0,"[karpovsky, method]",None,2,0,"[{'id': '53f44aa3dabfaee43ec8ec34', 'name': 'C...","[{'id': '562c496445cedb3398bde732', 'name': 'M...",1,1
1,"[audio-visual systems, handicapped aids, heari...","[error rate, connected word recognition proble...",23,11,"[{'id': '53f44a07dabfaeb22f4cee3a', 'name': 'R...","[{'id': '5484d790dabfae9b40133197', 'name': 'C...",2,4


### 3.2 Encoding not numeric features
For simple features that have few variant, we could opt to encode with a one hot encoding.
But for others we could need to opt for other choices, for exapmple for the keywords we would one hot encode only the most frequent, for authors, we will ...

In [71]:
cols_to_verify = ['lang_article', 'lang_ref', 'doc_type_article', 'doc_type_ref']

for c in cols_to_verify:
    n = train[c].nunique()
    print(f'The column "{c}" has {n} unique values.')

for c in ['keywords_article', 'keywords_ref']:
    unique_keys = set()
    for tags in df['keywords_article'].dropna():
        unique_keys.update(tags)
    n=len(unique_keys)
    print(f'The column "{c}" has {n} unique values.')

if os.path.exists(ID_REGISTRY_PATH):
    ids_registry = pd.read_parquet(ID_REGISTRY_PATH)
    print(f'The total number of authors {len(ids_registry)} (unique IDs).')

The column "lang_article" has 20 unique values.
The column "lang_ref" has 30 unique values.
The column "doc_type_article" has 2 unique values.
The column "doc_type_ref" has 2 unique values.
The column "keywords_article" has 243685 unique values.
The column "keywords_ref" has 243685 unique values.
The total number of authors 6729828 (unique IDs).


### Low cardinality Features : One-Hot Encoding (OHE)
For features with limited unique values, OHE provides a clear signal without a significant increase in dimensionality:
*   **Document Type**: With only 2 unique values for both article and reference (`doc_type_article` and `doc_type_ref`), OHE is used to represent the publication format efficiently.
*   **Language (Top 5 + Other)**: To manage the 20-30 unique languages, we encode the **Top 5 most frequent languages** via OHE and group the remaining "long tail" into a single "Other" category. This preserves the most significant linguistic patterns while reducing noise.


In [ ]:
train['lang_article'].value_counts()

lang_article
en    2054344
de       3138
fr       1464
es        324
pt        142
da         76
it         66
zh         30
nl         24
et         22
fi         16
pl         14
gl         10
ro          8
no          8
nn          6
ca          4
nb          4
lt          2
af          2
Name: count, dtype: int64

In [86]:
train, val, test = encode_categorical(train, val, test, ohe_cols=['lang_article', 'lang_ref', 'doc_type_article', 'doc_type_ref'], cols_top=['lang_article', 'lang_ref'], n_top=5)

### High-Cardinality Features: Hashing and Dropping
Handling columns like `authors` (over 6M unique IDs) and `keywords` (approx. 243K unique values) requires avoiding memory-intensive methods like One-Hot Encoding:
*   **Keywords (Feature Hashing)**: We utilize the **Hashing** to map the vast vocabulary of keywords into a fixed-dimensional space. This ensures the model captures semantic tags while maintaining a constant memory footprint and high parallelization efficiency.
*   **Authors (Dropped)**: Due to the extreme sparsity (more unique authors than total rows), individual author IDs are unlikely to generalize. We retain the numerical count (`n_authors`) to capture the "collaboration scale" while dropping the raw IDs to prevent overfitting.

> Since we don't want too much features, we decided to hash with 32 even if there is an high risk of collision.

In [94]:
train, val, test = hash_features(train, val, test, hash_cols=['keywords_article', 'keywords_ref'], n_features=16)

In [96]:
cols_to_drop = ['authors_article', 'authors_ref']

train.drop(columns=cols_to_drop, inplace=True, errors='ignore')
val.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test.drop(columns=cols_to_drop, inplace=True, errors='ignore')

print(f"Final feature count: {train.shape[1]}")

Final feature count: 56


## 4. Export Features

In [97]:
train.columns

Index(['year_article', 'n_citation_article', 'year_ref', 'is_reference_valid',
       'n_keywords_article', 'n_keywords_ref', 'n_authors_article',
       'n_authors_ref', 'lang_article_Other', 'lang_article_de',
       'lang_article_en', 'lang_article_es', 'lang_article_fr',
       'lang_article_pt', 'lang_ref_Other', 'lang_ref_de', 'lang_ref_en',
       'lang_ref_es', 'lang_ref_fr', 'lang_ref_pt',
       'doc_type_article_conference', 'doc_type_article_journal',
       'doc_type_ref_conference', 'doc_type_ref_journal',
       'keywords_article_hash_0', 'keywords_article_hash_1',
       'keywords_article_hash_2', 'keywords_article_hash_3',
       'keywords_article_hash_4', 'keywords_article_hash_5',
       'keywords_article_hash_6', 'keywords_article_hash_7',
       'keywords_article_hash_8', 'keywords_article_hash_9',
       'keywords_article_hash_10', 'keywords_article_hash_11',
       'keywords_article_hash_12', 'keywords_article_hash_13',
       'keywords_article_hash_14', 'keyword

In [99]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

train.to_parquet(OUTPUT_DIR / "train.parquet", index=False)
val.to_parquet(OUTPUT_DIR / "val.parquet", index=False)
test.to_parquet(OUTPUT_DIR / "test.parquet", index=False)